In [1]:
import pandas as pd

In [2]:
pois = pd.read_csv('Nodes/N-POIs.csv',dtype={'COD_POSTAL': 'str'})
pois['COD_POSTAL'].nunique()

38

In [3]:
demand = pd.read_csv('Data/mobilitat 2011 codis postals/dest-origen.csv',sep = ';',encoding='latin-1')
demand

,MUNIDEST,CPDESTI,ORIGEN,RAO,PERSONES
0,Badalona,08911,08911,Treball,2812
1,Badalona,08911,08912,Treball,1131
2,Badalona,08911,08913,Treball,673
3,Badalona,08911,08914,Treball,443
4,Badalona,08911,08915,Treball,824
...,...,...,...,...,...
4579,Terrassa,08228,08228,Estudi,683
4580,Terrassa,08228,Terrassa,Estudi,1222
4581,Terrassa,08228,Altres municipis,Estudi,156
4582,Terrassa,08228,Total,Estudi,1378


In [4]:
demand = pd.read_csv('Data/mobilitat 2011 codis postals/dest-origen.csv',sep = ';',encoding='latin-1')
demand = demand[demand['MUNIDEST'] == 'Barcelona']
demand = demand[demand['CPDESTI'] != '00Barcelona']
#demand = demand[demand['ORIGEN'] == 'Mataró']
demand['PERSONES'] = demand['PERSONES'].str.replace('..','0')
demand['PERSONES'] = demand['PERSONES'].astype(int)
demand_grouped = demand.groupby(['CPDESTI']).agg({'PERSONES':'sum'}).reset_index()
demand_grouped['PCT'] = demand_grouped['PERSONES'] / demand_grouped['PERSONES'].sum() * 100
demand_grouped

,CPDESTI,PERSONES,PCT
0,08001,134101,4.399208
1,08002,78734,2.582883
2,08003,96898,3.178756
3,08004,54355,1.783126
4,08005,103676,3.401110
5,08006,105535,3.462095
6,08007,87951,2.885248
7,08008,57394,1.882821
8,08009,61560,2.019487
9,08010,70064,2.298462


In [5]:
pois["N"] = pois.groupby("COD_POSTAL")["COD_POSTAL"].transform("size")

# Merge demand by postal code
pois = pois.merge(
    demand_grouped,
    left_on="COD_POSTAL",
    right_on="CPDESTI",
    how="left"
)

pois["PCT"] = pois["PCT"].div(pois["N"])
pois = pois.drop(columns=["N", "CPDESTI"])
pois['id'] = pois['id'].astype(str)

In [6]:
demand_full = pd.DataFrame()

# Mataro e11.1/2

In [7]:
FREQ = 4.4
H_SERVEI = 5
BUS_CAPACITY = 80
total_demand = FREQ * H_SERVEI * BUS_CAPACITY
total_demand

1760.0

In [8]:
pois

,id,geometry,barri,COD_POSTAL,PERSONES,PCT
0,Escola Universitaria Mediterrani,POINT (2.1522286 41.3803164),la Nova Esquerra de l'Eixample,08015,88794,1.456452
1,Centre d'Estudis Politècnics,POINT (2.1732364 41.388631),"Sant Pere, Santa Caterina i la Ribera",08010,70064,2.298462
2,Col·legi Major Universitari Lestonac,POINT (2.1660992 41.3928684),la Dreta de l'Eixample,08007,87951,0.961749
3,Escola Port,POINT (2.1955636 41.3925137),la Vila Olímpica del Poblenou,08005,103676,0.680222
4,Euroaula,POINT (2.1592325 41.3876943),l'Antiga Esquerra de l'Eixample,08011,38705,1.269725
...,...,...,...,...,...,...
83,Barcelona Activa (Ca N'Andalet),POINT (2.153488405116499 41.42697963945712),el Carmel,08032,53505,1.755241
84,Centre Català Comercial,POINT (2.1768176655253297 41.4288511516207),Vilapicina i la Torre Llobeta,08016,57664,1.891678
85,Nou Barris Activa,POINT (2.167723396618081 41.435811847133024),Can Peguera,08042,41256,1.353411
86,Mercabarna,POINT (2.115003621440011 41.33451392422421),la Marina del Prat Vermell,08040,53186,0.872388


In [9]:
demand_e11 = pois.copy()
demand_e11.drop(columns=['geometry','barri','COD_POSTAL','PERSONES'], inplace=True)
demand_e11['demand'] = (demand_e11['PCT'] *  total_demand / 100).round(0).astype(int)
demand_e11.drop(columns=['PCT'], inplace=True)
demand_e11 = (
    demand_e11
    .set_index("id")["demand"]
    .to_frame()
    .T
    .reset_index(drop=True)
)
demand_e11.index = ['E11']
demand_e11.sum(axis=1)

E11    1669
dtype: int64

In [10]:
demand_full = pd.concat([demand_full, demand_e11])
demand_full.to_csv('Data/Feed_optimization/Demand_matrix.csv')